In [1]:
import redis
import time
import base64
import json
import csv
import random
import statistics

import numpy as np
import pandas as pd

In [2]:
def sliding_window(data, window_size, step):
    for start_row in range(0, len(data) - window_size + 1, step):
        yield data[start_row:start_row + window_size]        

In [3]:
test = pd.read_csv("./data1/text_1.csv", header=None)
#test = pd.read_csv("text_2.csv", header=None)
#test = pd.read_csv("text_3.csv", header=None)

In [4]:
test = test.transpose()

In [7]:
humid = []
pm10 = []
pm25 = []
temp = []

for i in range(len(test)):
    val = test.values[i][0]
    val = val.replace("'", "")
    val = val.replace("b","",1)
    json_val = json.loads(val)
    
    res_payload = json_val['Payload']
    dec_res = base64.b64decode(res_payload)
    dec_res = dec_res.decode("UTF-8")
    str_test = dec_res.replace("'","\"")
    json_data = json.loads(str_test)
    
    print(json_data['event']['readings'])
    
    humid.append(json_data['event']['readings'][0]['objectValue']['humidity'])
    pm10.append(json_data['event']['readings'][0]['objectValue']['pm10'])
    pm25.append(json_data['event']['readings'][0]['objectValue']['pm25'])
    temp.append(json_data['event']['readings'][0]['objectValue']['temperature'])

[{'id': 'f11a16fc-acb0-43ae-bb6a-0252cc24ec57', 'origin': 1698804909747575732, 'deviceName': 'MD_05', 'resourceName': 'MicrodustNodeStatus', 'profileName': 'MD-Device-MQTT-Profile', 'valueType': 'Object', 'value': '', 'objectValue': {'device': {'data': [77, 68], 'type': 'Buffer'}, 'device_id': '56cb21c217805117', 'error': 0, 'humidity': 82, 'len': '45', 'msgid': {'data': [48, 50], 'type': 'Buffer'}, 'pm10': 30, 'pm25': 17, 'seqnum': '0001', 'temperature': 20, 'timestamp': '231101111459'}}]
[{'id': '9e4a9a98-abec-4841-9819-b0f40d9de689', 'origin': 1698804909751424096, 'deviceName': 'MD_07', 'resourceName': 'MicrodustNodeStatus', 'profileName': 'MD-Device-MQTT-Profile', 'valueType': 'Object', 'value': '', 'objectValue': {'device': {'data': [77, 68], 'type': 'Buffer'}, 'device_id': '1362f0b1b61b8322', 'error': 0, 'humidity': 85, 'len': '45', 'msgid': {'data': [48, 50], 'type': 'Buffer'}, 'pm10': 33, 'pm25': 17, 'seqnum': '0001', 'temperature': 19, 'timestamp': '231101111459'}}]
[{'id': '7

In [6]:
#절대값 차분
diff_humid = []
diff_pm10 = []
diff_pm25 = []
diff_temp = []

for i in range(len(humid)-1):
    diff_humid.append(abs(humid[i+1]-humid[i]))
    diff_pm10.append(abs(pm10[i+1]-pm10[i]))
    diff_pm25.append(abs(pm25[i+1]-pm25[i]))
    diff_temp.append(abs(temp[i+1]-temp[i]))

In [7]:
window_size = 50
step = 50

humid_window = list(sliding_window(diff_humid,window_size,step))
pm10_window = list(sliding_window(diff_pm10,window_size,step))
pm25_window = list(sliding_window(diff_pm25,window_size,step))
temp_window = list(sliding_window(diff_temp,window_size,step))

In [8]:
avg_dif_humid = []
avg_dif_pm10 = []
avg_dif_pm25 = []
avg_dif_temp = []

for i in range(len(humid_window)):
    avg_dif_humid.append(statistics.mean(humid_window[i]))
    avg_dif_pm10.append(statistics.mean(pm10_window[i]))
    avg_dif_pm25.append(statistics.mean(pm25_window[i]))
    avg_dif_temp.append(statistics.mean(temp_window[i]))

In [9]:
total_avdf_humid = statistics.mean(avg_dif_humid)
total_avdf_pm10 = statistics.mean(avg_dif_pm10)
total_avdf_pm25 = statistics.mean(avg_dif_pm25)
total_avdf_temp = statistics.mean(avg_dif_temp)

total_av = (total_avdf_humid + total_avdf_pm10 + total_avdf_pm25 + total_avdf_temp) / 4

In [10]:
#변수별 차분평균의 평균과 비교
cnt_val = []

for i in range(len(diff_temp)):
    total_av_val = (diff_humid[i] +diff_pm10[i] + diff_pm25[i] + diff_temp[i]) / 4
    if  total_av_val <= total_av:
        cnt_val.append(i)
print("제거된 트래픽 수 : ", len(cnt_val))

제거된 트래픽 수 :  522


In [11]:
#변수별 차분평균과 비교
cnt_val = []

for i in range(len(diff_temp)):
    if diff_humid[i] < total_avdf_humid and diff_pm10[i] < total_avdf_pm10 and diff_pm25[i] < total_avdf_pm25 and diff_temp[i] < total_avdf_temp:
        cnt_val.append(i)

print("제거된 트래픽 수 : ", len(cnt_val))

제거된 트래픽 수 :  82
